In [19]:
import os, gc, time, re
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
import xgboost as xgb
import joblib
from tqdm.notebook import tqdm

# Settings
FILE = "TwitterData_FE.csv"
N_SAMPLES = 30000      # or set None to use full dataset
RANDOM_STATE = 42
BERT_MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 64
BATCH_GPU = 8          # works well with RTX 3050
BATCH_CPU = 64
XGB_ESTIMATORS = 300
OUT_DIR = "models"

os.makedirs(OUT_DIR, exist_ok=True)


In [20]:
df = pd.read_csv(FILE)
df = df.dropna(subset=["Twitter_Account", "Tweet_text"])

if N_SAMPLES and N_SAMPLES < len(df):
    df = df.sample(n=N_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    df = df.reset_index(drop=True)

print("✅ Using rows:", len(df))
print(df.head(2))


✅ Using rows: 30000
  Twitter_User_Name Twitter_Account  \
0         Garry12gg       garry12gg   
1        Netflix US         netflix   

                            Twitter_User_Description             Tweet_id  \
0   I like to learn new things and play video games.   956425559566929000   
1  Since 5,456,245 of you asked, I'd like to form...  1054454611359090000   

   Tweet_created_at                                         Tweet_text  Label  \
0  25-01-2018 07:16                 RT @puddi: https://t.co/15QCSbU2Ls      0   
1  22-10-2018 19:29  @courtthebun ummm you are referring to THE win...      1   

   Word_Count  Url_Count  Retweet  ... QuesMark_Count  Exclamations_Count  \
0           3          1        1  ...              0                   0   
1          13          0        0  ...              0                   0   

   SpecialCharacters_Count  Nouns_Count  Pronouns_Count  Verb_Count  \
0                        2            5               0           0   
1           

In [21]:
num_cols = [
    "Word_Count","Url_Count","Retweet","Mentions_Count","Hashtags_Count",
    "QuesMark_Count","Exclamations_Count","SpecialCharacters_Count",
    "Nouns_Count","Pronouns_Count","Verb_Count","Adverb_Count",
    "Positive_Word_Ratio","Negative_Word_Ratio","Neutral_Word_Ratio"
]

X_numeric = df[num_cols].astype(float).reset_index(drop=True)
y = df["Label"].astype(int).reset_index(drop=True)
df["Twitter_User_Description"] = df["Twitter_User_Description"].fillna("")

df["combined_text"] = (
    df["Twitter_Account"].astype(str) + " " +
    df["Twitter_User_Description"].astype(str) + " " +
    df["Tweet_text"].astype(str)
)

print("Numeric features shape:", X_numeric.shape)
print("Sample combined text:\n", df["combined_text"].iloc[0])


Numeric features shape: (30000, 15)
Sample combined text:
 garry12gg I like to learn new things and play video games. RT @puddi: https://t.co/15QCSbU2Ls


In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
model = AutoModel.from_pretrained(BERT_MODEL_NAME).to(device)
model.eval()

if device.type == "cuda":
    model.half()
    print("⚡ Using GPU (half precision mode)")

def get_bert_embeddings(texts, batch_size=BATCH_GPU, max_len=MAX_LEN):
    all_emb = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating BERT embeddings"):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, truncation=True, padding=True, max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc).last_hidden_state
            emb = out.mean(dim=1).cpu().numpy()
        all_emb.append(emb)
        del enc, out, emb
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
    return np.vstack(all_emb)

texts = df["combined_text"].tolist()
t0 = time.time()
embeddings = get_bert_embeddings(texts)
t1 = time.time()

print(f"✅ Embeddings created: {embeddings.shape} in {round(t1 - t0, 2)}s")


⚡ Using GPU (half precision mode)


Generating BERT embeddings:   0%|          | 0/3750 [00:00<?, ?it/s]

✅ Embeddings created: (30000, 768) in 763.54s


In [23]:
X = np.hstack([X_numeric.values, embeddings])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

xgb_model = xgb.XGBClassifier(
    n_estimators=XGB_ESTIMATORS,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    predictor="cpu_predictor",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    eval_metric="logloss"
)

print("⚙️ Training XGBoost...")
t0 = time.time()
xgb_model.fit(X_train_scaled, y_train)
t1 = time.time()

print("✅ Trained in", round(t1 - t0, 1), "s")
y_pred = xgb_model.predict(X_test_scaled)
y_prob = xgb_model.predict_proba(X_test_scaled)[:, 1]

print("🎯 Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print(classification_report(y_test, y_pred))


⚙️ Training XGBoost...


C:\Users\rajat\Desktop\TwitterBotDetectionModel2\VENV\Lib\site-packages\xgboost\core.py:158: UserWarning: [22:45:20] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


✅ Trained in 59.6 s
🎯 Accuracy: 0.997
ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2905
           1       1.00      1.00      1.00      3095

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000



In [24]:
xgb_model.save_model(os.path.join(OUT_DIR, "xgboost_bot_detector.json"))
joblib.dump(scaler, os.path.join(OUT_DIR, "scaler.pkl"))
tokenizer.save_pretrained(os.path.join(OUT_DIR, "tokenizer"))
print("💾 Saved all model artifacts in /models/")


💾 Saved all model artifacts in /models/


In [25]:
booster = xgb.Booster()
booster.load_model("models/xgboost_bot_detector.json")

def extract_numeric_live(username, bio, tweet):
    return np.array([[
        len(tweet.split()), tweet.lower().count("http"), int(tweet.lower().startswith("rt ")),
        tweet.count("@"), tweet.count("#"), tweet.count("?"), tweet.count("!"),
        sum(1 for c in tweet if not c.isalnum() and not c.isspace()), 0,0,0,0, 0.0,0.0,1.0
    ]], dtype=float)

def predict_live(username, bio, tweet):
    text = f"{username} {bio} {tweet}"
    enc = tokenizer([text], return_tensors="pt", truncation=True, padding=True, max_length=64).to(device)
    with torch.no_grad():
        emb = model(**enc).last_hidden_state.mean(dim=1).cpu().numpy()
    num = extract_numeric_live(username, bio, tweet)
    feats = np.hstack([num, emb])
    feats_scaled = scaler.transform(feats)
    dmat = xgb.DMatrix(feats_scaled)
    prob = booster.predict(dmat)[0]
    return round(prob * 100, 2)

# test
print("Bot Probability:",
      predict_live("Ayesha_7", "Lost in the Music Ocean | Fan Account 💜", "NOT JUNGKOOK KNOWING ABOUT DEMON SLAYER ANIME 😂"),
      "%")


Bot Probability: 27.88 %
